# Prompt Optimization

This notebook shows how to use the **Prompt Optimization** feature to improve a prompt template for one or more target models, based on a labelled dataset and an optimization metric.

The SDK handles:
- Uploading your dataset to object storage
- Registering the AI Core configuration and execution
- Polling for completion
- Fetching the optimized prompt templates and evaluation scores

## Setup

In [ ]:
from gen_ai_hub.optimizations.client import OptimizationClient
from gen_ai_hub.optimizations.models import PromptOptimizationConfig
from dotenv import load_dotenv
import os

load_dotenv(override=True)

client = OptimizationClient(
    base_url=os.getenv("AICORE_BASE_URL"),
    auth_url=os.getenv("AICORE_AUTH_URL"),
    client_id=os.getenv("AICORE_CLIENT_ID"),
    client_secret=os.getenv("AICORE_CLIENT_SECRET"),
    resource_group=os.getenv("AICORE_RESOURCE_GROUP", "default"),
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
)

## One-Time Setup: Object Store Secret

The SDK needs an object store secret to upload your dataset and store output artifacts. Run this once per resource group — you can skip it on subsequent runs.

In [11]:
import json

AWS_S3_ENDPOINT = os.getenv("AWS_S3_ENDPOINT", "s3-eu-central-1.amazonaws.com")
AWS_BUCKET_ID = os.getenv("AWS_BUCKET_ID")
AWS_REGION = os.getenv("AWS_REGION", "eu-central-1")

default_secret_creds = {
    "data": {},
    "type": "S3",
    "pathPrefix": "sdkOutputFiles",
    "endpoint": AWS_S3_ENDPOINT,
    "bucket": AWS_BUCKET_ID,
    "region": AWS_REGION,
    "usehttps": "1",
}

response = client.setup(default_secret_body=default_secret_creds, replace_existing=True)

## Dataset Format

Your dataset must be a JSON file — a list of objects where each entry has exactly two top-level keys:

- **`fields`**: a dictionary of input variables. The keys must match the placeholder variables defined in your base prompt template (e.g. `{{?input}}`, `{{?page}}`, `{{?format_instructions}}`). Any number of fields is supported. Keys and values must be strings and must be consistent across all entries.
- **`answer`**: the expected output string (can be plain text or a JSON string)

```json
[
  {
    "fields": {
      "input": "Your input text here"
    },
    "answer": "Expected output"
  }
]
```

For prompts with multiple placeholders:

```json
[
  {
    "fields": {
      "page": "document text...",
      "format_instructions": "Return JSON with fields X, Y, Z"
    },
    "answer": "{\"x\": \"...\", \"y\": \"...\", \"z\": \"...\"}"
  }
]
```

## Defining the Optimization Config

`PromptOptimizationConfig` specifies everything the optimizer needs:

| Parameter | Required | Description |
|---|---|---|
| `dataset_path` | Yes* | Local path to your labelled JSON dataset. The SDK uploads it automatically. |
| `base_prompt` | Yes | Starting prompt template in `<scenario>/<name>:<version>` format |
| `target_models` | Yes | Models to optimize for, e.g. `["gpt-4o:2024-11-20"]` |
| `target_prompt_mapping` | Yes | Maps each target model to the output prompt name in the Prompt Registry |
| `optimization_metric` | Yes* | System metric to optimize for, e.g. `"JSON_Match"` |
| `custom_metric_id` | Yes* | Alternative to `optimization_metric` — use the ID of a custom metric |
| `base_model` | No | Reference model used internally during optimization |
| `include_few_shot_examples` | No | Whether to include few-shot examples (default: `False`) |
| `maximize` | No | Whether higher metric scores are better (default: `True`) |
| `correctness_cutoff` | No | Score threshold for correctness classification |
| `prompt_template_scope` | No | `"tenant"` or `"resourcegroup"` (default: `"tenant"`) |
| `prototype_mode` | No | Use as few as 3 samples for quick prototyping (default: `False`) |

*Either `dataset_path` or `artifact_id`+`dataset` must be provided. Either `optimization_metric` or `custom_metric_id` must be provided.

## Creating a Base Prompt Template

Before running optimization, you need a base prompt template in the Prompt Registry. The example below creates one — skip this if you already have one and just set `base_prompt` to your existing `<scenario>/<name>:<version>`.

In [12]:
from gen_ai_hub.prompt_registry.client import PromptTemplateClient
from gen_ai_hub.prompt_registry.models.prompt_template import PromptTemplate, PromptTemplateSpec

BASE_PROMPT_NAME = "my-base-prompt"
BASE_PROMPT_VERSION = "0.0.1"
BASE_PROMPT_SCENARIO = "genai-optimizations"

prompt_client = PromptTemplateClient(proxy_client=client._gen_ai_hub_proxy_client)

spec = PromptTemplateSpec(
    template=[
        PromptTemplate(role="system", content="You are a helpful assistant"),
        PromptTemplate(
            role="user",
            content=(
                "Giving the following message --- {{?input}} --- "
                "Extract and return a json with the following keys and values: "
                "- 'urgency' as one of `high`, `medium`, `low` "
                "- 'sentiment' as one of `negative`, `neutral`, `positive` "
                "- 'categories' Create a dictionary with categories as keys and boolean values (True/False), "
                "where the value indicates whether the category is one of the best matching support category tags from: "
                "`emergency_repair_services`, `routine_maintenance_requests`, `quality_and_safety_concerns`, "
                "`specialized_cleaning_services`, `general_inquiries`, `sustainability_and_environmental_practices`, "
                "`training_and_support_requests`, `cleaning_services_scheduling`, `customer_feedback_and_complaints`, "
                "`facility_management_issues` "
                "Your complete message should be a valid json string that can be read directly and only contain "
                "the keys mentioned in the list above. Never enclose it in ```json...```, no newlines, no unnecessary whitespaces."
            ),
        ),
    ],
    defaults={"input": ""},
    additional_fields={
        "modelParams": {"temperature": 0.7, "max_tokens": 100},
        "modelGroup": "chat",
    },
)

response = prompt_client.create_prompt_template(
    name=BASE_PROMPT_NAME,
    version=BASE_PROMPT_VERSION,
    scenario=BASE_PROMPT_SCENARIO,
    prompt_template_spec=spec,
)
base_prompt_id = response.id
base_prompt_ref = f"{BASE_PROMPT_SCENARIO}/{BASE_PROMPT_NAME}:{BASE_PROMPT_VERSION}"
print(f"Base prompt created: {base_prompt_id}")
print(f"Use as base_prompt: {base_prompt_ref}")

Base prompt created: 252f28cf-1c04-4438-b131-095dde3a6a5c
Use as base_prompt: genai-optimizations/my-base-prompt:0.0.1


In [17]:
# Option 1: local dataset file — SDK uploads it to S3 automatically.
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), "../../.."))
dataset_path = os.path.join(PROJECT_ROOT, "integration_tests/optimizations/po_dataset.json")

optimization_config = PromptOptimizationConfig(
    dataset_path=dataset_path,
    base_prompt=base_prompt_ref,
    target_models=["gemini-2.5-pro:001"],
    target_prompt_mapping={
        "gemini-2.5-pro:001": "base-prompt-custom-gemini-25-pro:0.0.1",
    },
    optimization_metric="JSON_Match",
)

In [18]:
# Option 2: reuse an already-uploaded artifact — skips the S3 upload.
#
# optimization_config = PromptOptimizationConfig(
#     artifact_id="<artifact_id from previous run>",
#     dataset="testdata/<filename>.json",
#     base_prompt="genai-optimizations/my-base-prompt:1.0.0",
#     target_models=["gpt-4o:2024-11-20"],
#     target_prompt_mapping={
#         "gpt-4o:2024-11-20": "my-optimized-prompt:0.0.1",
#     },
#     optimization_metric="JSON_Match",
# )

## Running the Optimization

`client.optimize()` validates the config, uploads the dataset, registers the AI Core execution, and returns an `OptimizationRun` object immediately. The job runs asynchronously in the background.

In [19]:
optimization_run = client.optimize(optimization_config)
print(f"Optimization started. Execution ID: {optimization_run._run_context.execution_id}")

Optimization started. Execution ID: e001e2b8bdc9d575


## Wait for Completion

Use `wait_for_completion()` to block until the job finishes. You can also call `get_current_status()` at any point to check progress without blocking.

In [21]:
optimization_run.wait_for_completion(timeout=3600)  # timeout in seconds, default is 1 hour
print(f"Status: {optimization_run.get_current_status()}")

Status: Status.COMPLETED


## Debugging

If the job fails, use these helpers to investigate.

In [ ]:
import json

# Structured summary of which step failed and why
debug_info = optimization_run.get_debug_info()
print("Debug info:", json.dumps(debug_info, indent=4, default=str))

# Full execution logs
debug_logs = optimization_run.get_debug_logs()
print("Logs:", json.dumps(debug_logs, indent=4, default=str))

## Viewing Results

`results()` returns an `OptimizationResults` object with:

- **`metrics`** — pre/post evaluation scores per model from the tracking service
- **`prompts`** — the optimized prompt template from the Prompt Registry for each target model

Printing the object shows a formatted summary of both.

> **Note:** If the optimized prompt could not be fetched from the Prompt Registry, the results will still contain the metrics but the prompt section will be omitted. In that case, you can retrieve the optimized prompt from the execution logs using `get_debug_logs()`

In [23]:
results = optimization_run.results()
print(results)

OptimizationResults
══════════════════════════════════════════════════════════════════════

  Score Summary
  ┌────────────────────┬──────────┬───────┬───────┬─────────────┐
  │       Model        │ Baseline │  Pre  │  Post │ Improvement │
  ├────────────────────┼──────────┼───────┼───────┼─────────────┤
  │ gemini-2.5-pro:001 │    –     │ 0.910 │ 0.963 │   ▲ +5.8%   │
  └────────────────────┴──────────┴───────┴───────┴─────────────┘

  Evaluation Details  ─  gemini-2.5-pro:001
  ┌────────────┬───────┬───────┐
  │   Metric   │  Pre  │  Post │
  ├────────────┼───────┼───────┤
  │ f1         │ 0.910 │ 0.963 │
  │ tp         │   273 │   289 │
  │ llm_p      │   300 │   300 │
  │ recall     │ 0.910 │ 0.963 │
  │ ground_p   │   300 │   300 │
  │ precision  │ 0.910 │ 0.963 │
  │ is_correct │    23 │    25 │
  └────────────┴───────┴───────┘

  Custom Info  ─  gemini-2.5-pro:001
    llm_request_metrics: {'provider_request_counts': [{'provider': {'model_name': 'gpt-5', 'model_params': {'reasoni

## Cleanup

Delete the base prompt template created earlier.

In [24]:
prompt_client.delete_prompt_template_by_id(base_prompt_id)
print(f"Base prompt deleted: {base_prompt_id}")

Base prompt deleted: 252f28cf-1c04-4438-b131-095dde3a6a5c
